In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings

warnings.filterwarnings("ignore")

In [8]:
glass = pd.read_csv("Glass.csv")
X = glass.drop('Type', axis=1)
le = LabelEncoder()
y = le.fit_transform( glass['Type'] )
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5, 5, 5, 5, 5, 5, 5, 5,
       5, 5, 5, 5, 5, 5, 5, 5, 5, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3])

In [3]:
le.classes_

array(['building_windows_float_processed',
       'building_windows_non_float_processed', 'containers', 'headlamps',
       'tableware', 'vehicle_windows_float_processed'], dtype=object)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25,
                                                    stratify=y)

## Feature Scaling

In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

# Compare penalties using SAGA solver

In [6]:
solver = 'saga'

penalties = [
    'l2',
    'l1',
    'elasticnet',
    None
]

scores = []


for p in penalties:

    if p == 'elasticnet':

        lr = LogisticRegression(
            solver=solver,
            penalty=p,
            l1_ratio=0.5,
            max_iter=5000,
            random_state=25
        )

    else:

        lr = LogisticRegression(
            solver=solver,
            penalty=p,
            max_iter=5000,
            random_state=25
        )


    lr.fit(X_train_scaled,y_train)
    y_pred = lr.predict(X_test_scaled)
    scores.append([p,accuracy_score(y_test, y_pred)])

df_scores = pd.DataFrame(scores,columns=['Penalty','Accuracy'])
print(df_scores.sort_values('Accuracy',ascending=False))


      Penalty  Accuracy
1          l1  0.661538
3         NaN  0.661538
2  elasticnet  0.646154
0          l2  0.630769


In [9]:
solvers = [
    'sag',
    'newton-cg',
    'newton-cholesky',
    'lbfgs'
]


penalties = [
    'l2',
    None
]
scores = []
for p in penalties:

    for s in solvers:


        lr = LogisticRegression(
            solver=s,
            penalty=p,
            max_iter=5000,
            random_state=25
        )


        lr.fit(
            X_train_scaled,
            y_train
        )


        y_pred = lr.predict(
            X_test_scaled
        )


        scores.append(
            [
                p,
                s,
                accuracy_score(
                    y_test,
                    y_pred
                )
            ]
        )

df_scores = pd.DataFrame(
    scores,
    columns=[
        'Penalty',
        'Solver',
        'Accuracy'
    ]
)

print(
    df_scores.sort_values(
        'Accuracy',
        ascending=False
    )
)


  Penalty           Solver  Accuracy
5     NaN        newton-cg  0.661538
7     NaN            lbfgs  0.661538
4     NaN              sag  0.661538
6     NaN  newton-cholesky  0.646154
3      l2            lbfgs  0.630769
2      l2  newton-cholesky  0.630769
1      l2        newton-cg  0.630769
0      l2              sag  0.630769


In [10]:
lr = LogisticRegression(solver='newton-cg',penalty='l2',max_iter=5000,random_state=25)
lr.fit(X_train_scaled,y_train)
# Prediction probability
y_pred_prob = lr.predict_proba(X_test_scaled)
print(y_pred_prob)

[[7.17178996e-01 1.39190694e-01 1.46541293e-04 3.84401297e-04
  1.77968056e-03 1.41319686e-01]
 [3.04912082e-01 5.45191683e-01 2.28580442e-02 1.62964309e-02
  1.54897909e-02 9.52519691e-02]
 [4.31098025e-01 4.77745697e-01 2.22773870e-02 1.23087090e-02
  8.47051148e-03 4.80996707e-02]
 [4.53827039e-01 4.57083917e-01 2.00294733e-02 5.50500062e-03
  5.65192776e-03 5.79026423e-02]
 [3.80150129e-01 4.72680905e-01 1.73609535e-02 1.71822220e-02
  2.08262935e-02 9.17994971e-02]
 [6.62940500e-01 2.39006833e-01 2.06007368e-03 1.80628847e-03
  3.24288155e-03 9.09434231e-02]
 [7.46102564e-01 1.63072734e-01 5.24795264e-04 1.88068498e-03
  5.81891872e-03 8.26003027e-02]
 [5.81354666e-01 2.77601177e-01 3.60237013e-03 4.32860004e-03
  1.34236344e-02 1.19689552e-01]
 [4.60368616e-01 4.30669310e-01 1.57241034e-02 7.10543707e-03
  8.75704570e-03 7.73754876e-02]
 [4.18393310e-01 4.45405124e-01 1.24305147e-02 1.12334161e-02
  2.22667357e-02 9.02708992e-02]
 [8.89715244e-02 4.16037887e-01 3.19165089e-01 8.5

In [11]:
# Prediction
y_pred = lr.predict(X_test_scaled)
print(y_pred)
# Accuracy
print(accuracy_score(y_test,y_pred))

[0 1 1 1 1 0 0 0 0 1 1 0 1 4 3 0 3 1 1 4 0 0 0 0 0 0 0 0 2 0 1 2 3 1 1 0 0
 2 2 1 1 0 0 0 3 1 1 1 1 3 1 1 0 0 1 1 3 3 0 0 3 1 5 0 1]
0.6307692307692307


In [14]:
# Confusion Matrix
cm = confusion_matrix(y_test,y_pred)
print(cm)

[[15  5  0  0  0  1]
 [ 7 15  1  0  0  0]
 [ 0  2  2  0  0  0]
 [ 0  1  1  7  0  0]
 [ 0  0  0  1  2  0]
 [ 4  1  0  0  0  0]]


In [15]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues', xticklabels=le.classes_,yticklabels=le.classes_)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Logistic Regression")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.show()

In [ ]:
# Classification Report with class names

print(classification_report(y_test,y_pred,target_names=le.classes_))